In [1]:
import polars as pl
from scipy.sparse import csr_matrix                                                                                                                                                                                                                      
import mlflow
articles = pl.read_csv("../data/raw/articles.csv")
customers = pl.read_csv("../data/raw/customers.csv")
transactions = pl.scan_csv("../data/raw/transactions_train.csv")
mlflow.set_tracking_uri("file:///home/nico/nico_mle/mlruns")
mlflow.set_experiment("hm-recsys")

/home/nico/nico_mle/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///home/nico/nico_mle/mlruns/369336250363885497', creation_time=1776346502972, experiment_id='369336250363885497', last_update_time=1776346502972, lifecycle_stage='active', name='hm-recsys', tags={}, trace_location=None, workspace='default'>

In [2]:
activity = (
    transactions
    .group_by("customer_id")
    .agg(pl.len().alias("count"))
    .with_columns([
        (pl.col("count") > 500).cast(pl.Int8).alias("high_activity_flag"),
        (pl.col("count") < 5).cast(pl.Int8).alias("low_activity_flag"),
    ])
    .collect()
)

In [3]:
item_activity = (
    transactions
    .group_by("article_id")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") < 5).cast(pl.Int8).alias("low_activity_item")
    )
    .collect()
)

In [4]:
transactions_filtered = (
    transactions
    .join(
        activity
            .filter((pl.col("high_activity_flag") == 0) & (pl.col("low_activity_flag") == 0))
            .select("customer_id")
            .lazy(),
        on="customer_id", how="inner"
    )
    .join(
        item_activity
            .filter(pl.col("low_activity_item") == 0)
            .select("article_id")
            .lazy(),
        on="article_id", how="inner"
    )
    .collect()
)

print(transactions_filtered.shape)  # expect ~(30_480_760, 5)

(30454020, 5)


In [5]:
transactions_sorted = transactions_filtered.sort("t_dat", "article_id")
del transactions_filtered  # free ~3GB immediately after sort
transactions_sorted.head(5)

t_dat,customer_id,article_id,price,sales_channel_id
str,str,i64,f64,i64
"""2018-09-20""","""3329634c9451f438049c5ebb4e77d0…",108775015,0.008458,1
"""2018-09-20""","""4ff6618b35d189c84a07d56392d4e3…",108775015,0.008458,1
"""2018-09-20""","""20ccafd82d923baecf1fb8705d459e…",108775015,0.008458,1
"""2018-09-20""","""7e1030796c41a025996e90435ee5a9…",108775015,0.008458,2
"""2018-09-20""","""7e1030796c41a025996e90435ee5a9…",108775015,0.008458,2


In [6]:
# Get integer row index of last transaction per customer (1 index per customer, ~1.3M entries)
last_row_idx = (
    transactions_sorted
    .with_row_index("_i")
    .group_by("customer_id")
    .agg(pl.col("_i").last())
    ["_i"]
    .to_list()
)

test = transactions_sorted[last_row_idx]

train = (
    transactions_sorted
    .with_row_index("_i")
    .filter(~pl.col("_i").is_in(last_row_idx))
    .drop("_i")
)
del transactions_sorted

print("Train transactions:", len(train))
print("Test transactions:", len(test))
test.head(5)

Train transactions: 29528936
Test transactions: 925084


t_dat,customer_id,article_id,price,sales_channel_id
str,str,i64,f64,i64
"""2020-07-30""","""8e399e4f7866e5c70f41a1cecdee61…",915027001,0.009932,1
"""2020-01-31""","""1e1285a44d11e2e5203bffafbab83a…",835044007,0.069475,1
"""2020-07-28""","""4093171e17c147d8e7c1a8cd3587a0…",887681003,0.022017,2
"""2019-07-24""","""81ddf8336c7757ad376cc7c6322ced…",684341002,0.008119,2
"""2020-08-26""","""7e5318b17a401bcf2b793f1c1e269a…",894788001,0.016932,1


In [7]:
agg_train = train.group_by(["customer_id","article_id"]).agg(pl.len().alias("purchase_count"))

In [8]:
agg_train_pd = agg_train.to_pandas()                                                                                                                                                                                                                                 
agg_train_pd["customer_id"] = agg_train_pd["customer_id"].astype("category")                                                                                                                                                                                       
agg_train_pd["article_id"]  = agg_train_pd["article_id"].astype("category") 
                                                                                                                                                                                                                                                         
row  = agg_train_pd["customer_id"].cat.codes.to_numpy()
col  = agg_train_pd["article_id"].cat.codes.to_numpy()                                                                                                                                                                                                         
agg_train_data = agg_train_pd["purchase_count"].to_numpy(dtype="int32")                                                                                                                                                                                                
                                                                                                                                                                                                                                                         
customer_idx_train = agg_train_pd["customer_id"].cat.categories  # int → customer_id                                                                                                                                                                                 
article_idx_train  = agg_train_pd["article_id"].cat.categories   # int → article_id                                                                                                                                                                                  
                                                                                                                                                                                                                                                         
n_users, n_items = customer_idx_train.size, article_idx_train.size                                                                                                                                                                                                   
user_item_train = csr_matrix((agg_train_data, (row, col)), shape=(n_users, n_items), dtype="int32")

In [9]:
print("Train matrix shape:", user_item_train.shape)

Train matrix shape: (925082, 91863)


In [10]:
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

with mlflow.start_run(run_name="ALS_eval"):
    model_als = AlternatingLeastSquares(
        factors=50, regularization=0.1, iterations=20, random_state=42
    )
    mlflow.log_params({"model": "ALS", "factors": 50, "regularization": 0.1, "iterations": 20})
    model_als.fit(user_item_train)

with mlflow.start_run(run_name="BPR_eval"):
    model_bpr = BayesianPersonalizedRanking(
        factors=50, learning_rate=0.01, regularization=0.01, iterations=100, random_state=42
    )
    mlflow.log_params({"model": "BPR", "factors": 50, "learning_rate": 0.01, "regularization": 0.01, "iterations": 100})
    model_bpr.fit(user_item_train)

/home/nico/nico_mle/venv/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [11]:
def evaluate(model, user_item_matrix, test_df, customer_idx, article_idx, K=10, n_users=1000):
    customer_map = {v: i for i, v in enumerate(customer_idx)}
    article_map = {v: i for i, v in enumerate(article_idx)}

    test_users = test_df["customer_id"].unique().to_list()
    test_users = [u for u in test_users if u in customer_map][:n_users]

    precisions, recalls = [], []

    for customer_id in test_users:
        user_idx = customer_map[customer_id]

        ground_truth = test_df.filter(
            pl.col("customer_id") == customer_id
        )["article_id"].to_list()
        ground_truth = [a for a in ground_truth if a in article_map]

        if not ground_truth:
            continue

        ids, _ = model.recommend(
            user_idx, user_item_matrix[user_idx],
            N=K, filter_already_liked_items=True
        )
        recommended = article_idx[ids].tolist()

        hits = len(set(recommended) & set(ground_truth))
        precisions.append(hits / K)
        recalls.append(hits / len(ground_truth))

    return {
        "precision_at_k": round(sum(precisions) / len(precisions), 4),
        "recall_at_k": round(sum(recalls) / len(recalls), 4)
    }


# Evaluate and log
K = 10

with mlflow.start_run(run_name="ALS_eval"):
    metrics = evaluate(model_als, user_item_train, test, customer_idx_train, article_idx_train, K=K)
    mlflow.log_param("K", K)
    mlflow.log_metrics(metrics)
    print("ALS:", metrics)

with mlflow.start_run(run_name="BPR_eval"):
    metrics = evaluate(model_bpr, user_item_train, test, customer_idx_train, article_idx_train, K=K)
    mlflow.log_param("K", K)
    mlflow.log_metrics(metrics)
    print("BPR:", metrics)

ALS: {'precision_at_k': 0.001, 'recall_at_k': 0.01}
BPR: {'precision_at_k': 0.0009, 'recall_at_k': 0.009}
